# Using Gemini and GigaChat and creation of prose-to-poetry dataset

In this notebook you can see the code for evaluation Gigachat and Gemini and creation of prose-to-poetry dataset 

## General functions and imports

In [1]:
from time import sleep
import pandas as pd

system_instruction = '''Вы – талантливый поэт, создающий русскую поэзию. При преобразовании прозы в стихотворение соблюдайте следующие правила:\n''' \
                    '''1. Рифмовка: используйте заданную схему рифм.\n''' \
                    '''2. Размер: пишите в указанном метре и соблюдайте структуру чередования ударных и безударных слогов.\n''' \
                    '''3. Объём: стихотворение должно содержать ровно 4 строки.\n''' \
                    '''4. Содержание: сохраняйте ключевые образы, эмоции, детали описаний и действия из исходного текста.\n''' \
                    '''5. Выразительность: сделайте текст поэтичным и насыщенным.\n''' \
                    '''6. Формат: в ответе должно быть исключительно само стихотворение без комментариев.\n'''

system_instruction_generate = '''Вы – талантливый поэт, создающий русскую поэзию. При написании стихотворения соблюдайте следующие правила:\n''' \
                    '''1. Рифмовка: используйте заданную схему рифм.\n''' \
                    '''2. Размер: пишите в указанном метре и соблюдайте структуру чередования ударных и безударных слогов.\n''' \
                    '''3. Объём: стихотворение должно содержать ровно 4 строки.\n''' \
                    '''5. Выразительность: сделайте текст поэтичным и насыщенным.\n''' \
                    '''6. Формат: в ответе должно быть исключительно само стихотворение без комментариев.\n'''

system_instruction_inv = '''Вы – профессиональный литературный редактор, специалист по адаптации русской поэзии в художественную прозу. Преобразуйте данное стихотворение в прозаический текст, соблюдая следующие правила:\n''' \
        '''1. Сохранение смысла: передайте основное содержание, образы, идеи и настроение стихотворения.\n''' \
        '''2. Объём: длина полученного текста должна соответствовать объёму исходного стихотворения.\n''' \
        '''3. Грамотная стилистика: используйте литературный язык, избегая слишком сухого или официального тона.\n''' \
        '''4. Связность и плавность: стройте текст логично и последовательно, обеспечивая естественный поток мысли.\n''' \
        '''5. Передача эмоций: сохраняйте эмоциональную окраску, экспрессию и художественные детали оригинала.\n''' \
        '''6. Избегание ритма и рифмы: уберите поэтические конструкции, но при необходимости используйте выразительные средства прозы (метафоры, эпитеты и т. д.).\n''' \
        '''7. Формат: в ответе должно быть исключительно результат без комментариев.\n'''

meters = {
    'ямб': 'ямбический - чередуются ударные и безударные слоги, первый слог строки безударный',
    'iambos': 'ямбический - чередуются ударные и безударные слоги, первый слог строки безударный',
    'choreios': 'хорей - чередуются ударные и безударные слоги, первый слог строки ударный',
    'dolnik3': 'дольник - стихотворный размер с переменным количеством безударных слогов между ударными',
    'amphibrachys': 'амфибрахий - трехсложный размер, где ударение падает на второй слог',
    'anapaistos': 'анапест - трехсложный размер, где ударение падает на третий слог',
    'daktylos': 'дактиль - трехсложный размер, где ударение падает на первый слог',
    'dolnik2': 'дольник - стихотворный размер с переменным количеством безударных слогов между ударными',
}

def get_prompt(text, scheme='ABAB', meter='ямб'):
    if text is None:
        return f'''Напиши четверостишие с параметрами:\n Рифмовка: {scheme}\n Размер: {meters[meter]}\n'''
    return f'''Преобразуй прозу в четверостишие с параметрами:\n Рифмовка: {scheme}\n Размер: {meters[meter]}\n Исходный текст: {text}'''


In [2]:
import ast

def use_model_batch(func, texts, from_id=0):
    answers = {}
    for i, (id, text) in enumerate(texts.items()):
        if i // 15 < from_id:
            continue
        if i % 15 == 0 and len(answers) != 0:
            yield pd.Series(answers)
            answers = {}
        answers[id] = func(text)
    if len(answers) != 0:
        yield pd.Series(answers)

def generate_model_answers(model_func, file_path='test_text.txt', from_id=0, from_index=0, to_index=-1):
    inputs = {}
    with open(file_path, 'r') as file:
        for i, line in enumerate(file.readlines()):
            if i >= from_index and (i < to_index or to_index == -1):
                inputs[i] = line
    answers = []
    i = from_id
    for answer in use_model_batch(model_func, inputs, from_id=from_id):
        answer.to_csv(f'answers{i}.txt')
        answers.append(answer)
        print(i)
        i += 1
    return pd.concat(answers)

def test_model(model_func, file_path='prosa_test_text.csv', from_id=0):
    inputs = {}
    df = pd.read_csv(file_path)
    for index, row in df.iterrows():
        inputs[index] = row
    answers = []
    i = from_id
    for answer in use_model_batch(model_func, inputs, from_id=from_id):
        answer.to_csv(f'answers{i}.txt')
        answers.append(answer)
        print(i)
        i += 1
    return pd.concat(answers)

def generate_model_answers_inv(model_func, file_path='stanzas_test.csv', from_id=0, from_index=0, to_index=-1):
    inputs = {}
    df = pd.read_csv(file_path)
    for index, row in df.iterrows():
        st = ast.literal_eval(row['stanzas'])
        if index >= from_index and (index < to_index or to_index == -1):
            inputs[index] = '\n'.join(st)
    answers = []
    i = from_id
    for answer in use_model_batch(model_func, inputs, from_id=from_id):
        answer.to_csv(f'answers{i}.txt')
        answers.append(answer)
        print(i)
        i += 1
    return pd.concat(answers)

## Configuring

### Gemini

Configuring of Gemini API. Use yor API key instead of `your_key`.

In [ ]:
%pip install -q -U google-generativeai

In [ ]:
import google.generativeai as genai

genai.configure(api_key="your_key")

### Gigachat

Configuring Gigachat API. Use your token instead of `your_token` in `Authorization`.

In [ ]:
!curl -k "https://gu-st.ru/content/Other/doc/russian_trusted_root_ca.cer" -w "\n" >> russian_trusted_root_ca.cer

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2088    0  2088    0     0   1701      0 --:--:--  0:00:01 --:--:--  1701


In [ ]:
import requests
import uuid
import json

def get_access_token():
    url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"

    payload='scope=GIGACHAT_API_PERS'
    # Создание случайного UUID
    random_uuid = uuid.uuid4()
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
        'Accept': 'application/json',
        'RqUID': str(random_uuid),
        'Authorization': 'Basic your_token'
    }

    response = requests.request("POST", url, headers=headers, data=payload, verify='russian_trusted_root_ca.cer')
    return response.json()["access_token"]

access_token = get_access_token()

Check available models.

In [ ]:
url = "https://gigachat.devices.sberbank.ru/api/v1/models"

payload={}
headers = {
  'Accept': 'application/json',
  'Authorization': f'Bearer {access_token}'
}

response = requests.request("GET", url, headers=headers, data=payload, verify='russian_trusted_root_ca.cer')

response.json()

{'object': 'list',
 'data': [{'id': 'GigaChat',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-2',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-2-Max',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-2-Max-preview',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-2-Pro',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-2-Pro-preview',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-2-preview',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-Max',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-Max-preview',
   'object': 'model',
   'owned_by': 'salutedevices',
   'type': 'chat'},
  {'id': 'GigaChat-Plus',
   'object': 'model',
  

## Evaluation

### Gemini

This code evaluates Gemini-2.0-Flash on `prosa_test_text.csv` file.

In [ ]:
def use_gemini_row(row):
    while True:
        try:
            model = genai.GenerativeModel(
                        model_name="gemini-2.5-flash",
                        system_instruction=system_instruction)
            response = model.generate_content(get_prompt(row['text'], row['rhyme_scheme'], row['meter']))
            return response.text
        except:
            print('error')
            sleep(30)

In [ ]:
answers = test_model(use_gemini_row, file_path='prosa_test_text.csv', from_id=0)
answers.to_csv('gemini.csv', index=False)

### Gigachat

Evaluate Gigachat on file `prosa_test_text.csv`.

In [ ]:
url = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"

def use_gigachat_row(row, model='GigaChat-2'):
    global access_token
    headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': f'Bearer {access_token}'
    }
    payload = json.dumps({
        "model": model,
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": get_prompt(row['text'], row['rhyme_scheme'], row['meter'])}
        ],
        "stream": False,
        "repetition_penalty": 1
    })
    try:
        response = requests.request("POST", url, headers=headers, data=payload, verify='russian_trusted_root_ca.cer')
        return response.json()['choices'][0]['message']['content']
    except:
        print('error')
        access_token = get_access_token()
        headers = {
        'Content-Type': 'application/json',
        'Accept': 'application/json',
        'Authorization': f'Bearer {access_token}'
        }
        response = requests.request("POST", url, headers=headers, data=payload, verify='russian_trusted_root_ca.cer')
        return response.json()['choices'][0]['message']['content']

In [ ]:
answers = test_model(lambda row: use_gigachat_row(row, model='GigaChat-2'), file_path='prosa_test_text.csv', from_id=0)
answers.to_csv('gigachat.csv', index=False)

## Creation of prose-poetry dataset

### First creation using Gigachat

Creation of prose-to-poetry dataset. Uses Gigachat to create prose from `trainset.csv` and `testset.csv` files.

In [ ]:
url = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"

def use_gigachat_inv(text, model='GigaChat-2'):
    global access_token
    headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': f'Bearer {access_token}'
    }
    payload = json.dumps({
        "model": model,
        "messages": [
            {"role": "system", "content": system_instruction_inv},
            {"role": "user", "content": text}
        ],
        "stream": False,
        "repetition_penalty": 1
    })
    try:
        response = requests.request("POST", url, headers=headers, data=payload, verify='russian_trusted_root_ca.cer')
        return response.json()['choices'][0]['message']['content']
    except:
        print('error')
        access_token = get_access_token()
        headers = {
        'Content-Type': 'application/json',
        'Accept': 'application/json',
        'Authorization': f'Bearer {access_token}'
        }
        response = requests.request("POST", url, headers=headers, data=payload, verify='russian_trusted_root_ca.cer')
        return response.json()['choices'][0]['message']['content']

Creating prosa for trainset

In [ ]:
answers = generate_model_answers_inv(use_gigachat_inv,
                                     file_path='trainset.csv',
                                     from_id=0)
answers.to_csv('gigachat_train_inv.csv', index=False)

In [ ]:
ans = pd.read_csv('gigachat_train_inv.csv')[['0']].rename({'0': 'input'}, axis=1)
ans['index'] = range(1000, 11000)
ans = ans.set_index('index')
inputs = pd.read_csv('trainset.csv')
df = pd.concat([inputs, ans], axis=1)
df = df[~df['input'].isna()]
df.to_csv('trainset.csv')

Creating prosa for testset.

In [ ]:
answers = generate_model_answers_inv(use_gigachat_inv,
                                     file_path='testset.csv',
                                     from_id=0)
answers.to_csv('gigachat_test_inv.csv', index=False)

In [ ]:
ans = pd.read_csv('gigachat_test_inv.csv')[['0']].rename({'0': 'input'}, axis=1)
ans['index'] = range(0, 1000)
ans = ans.set_index('index')
inputs = pd.read_csv('testset.csv')
df = pd.concat([inputs, ans], axis=1)
df = df[~df['input'].isna()]
df.to_csv('testset.csv')

### Recreation of bad examples

The following cells execute the data cleaning and re-generation pipeline using Gemini. The operations are applied to the file specified by the `path` variable (currently `'dataset/trainset.csv'`). Note that this identical pipeline was also executed for the test set (`'dataset/testset.csv'`).

Set up the environment

In [ ]:
import sys
sys.path.append('c:\\HSE\\project_poetry_2\\ProjectPoetryRL\\')
sys.path.append('c:\\HSE\\project_poetry_2\\ProjectPoetryRL\\prose-to-poetry')
%cd ..

Import libs

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import random
import torch.nn.functional as F

from metrics import encode_sent

Define new instruction to convert poem to prose.

In [ ]:
sys_inst = '''Вы – профессиональный литературный редактор, специалист по адаптации русской поэзии в художественную прозу. Преобразуйте данные стихотворения в прозаический текст, соблюдая следующие правила: 
1. Сохранение смысла: передайте основное содержание, образы, идеи и настроение стихотворения. Сохрани все имена и названия. 
2. Объём: длина полученного текста должна соответствовать объёму исходного стихотворения. Она не должна сильно увеличиватся 
3. Изменение: замени слова на синонимы и поменяй порядок слов. Убери стихотворные особенности языка.
4. Прямая речь: сохрани прямую речь там, где она использована и повествование от первого лица, если оно было в исходном стихе. 
5. Формат: для каждого стиха напиши index индекс, REAL: сам стих и PROSE: полученную прозу. НЕ пиши иные комментарии.

Пример:
index 2561
REAL: Кощеева; а как найти ту смерть, и я
Того не ведаю; об этом Баба
Яга одна сказать лишь может. Ты,
Иван-царевич, должен эту Бабу
PROSE: Смерть Кощея спрятана неизвестно где, и я сама не знаю, как её найти; об этом может рассказать только Баба-Яга. Ты, Иван-царевич, должен отправиться к этой Бабе

Стихи:
'''

Read dataset and encode sentences.

In [ ]:
path = 'dataset/trainset.csv'

# загрузка датасета
df = pd.read_csv(path)

# функция: из списка строк делаем один текст
def join_stanzas(stanzas_str):
    try:
        stanzas = ast.literal_eval(stanzas_str)
        return "\n".join(stanzas)
    except:
        return stanzas_str  # если уже строка

# готовим данные
proses = df['input'].tolist()
poems = [join_stanzas(s) for s in df['stanzas']]

# считаем эмбеддинги 
prose_emb = encode_sent(proses)
poem_emb = encode_sent(poems)

The logic below evaluates each row against a set of formatting and semantic rules to flag low-quality pairs. Since sending all failed rows at once is impractical, this cell is designed to extract and print the invalid data **in segments of 25 samples**. The `ind` variable controls which specific batch is generated and printed, preparing manageable blocks of text to be passed over to Gemini.

In [ ]:
bad = 0
total = 0

df = pd.read_csv(path)
df['len'] = [len('\n'.join(ast.literal_eval(x))) for x in df['stanzas']]
df['len_input'] = [len(x) for x in df['input']]

count = 0

def clean_text(text):
    text = re.sub(r"[^\w\s.,;:!?\"'«»()\[\]\-–—…]", "", text)
    return text

def jaccard_similarity(a, b):
    a_set = set(a.lower().split())
    b_set = set(b.lower().split())
    return len(a_set & b_set) / len(a_set | b_set)

errs ={
    'jaccard': 0,
    'len-': 0,
    'st in input': 0,
    'nan': 0,
    'len input': 0,
    'len/<0.75': 0,
    'len/>2.5': 0,
    'bad_emb': 0,
}

def check(row, i):
    st = ast.literal_eval(row['stanzas'])
    if jaccard_similarity(row['input'], '\n'.join(st)) > 0.7:
        errs['jaccard'] += 1
        return True, 'jaccard'
    if abs(row['len'] -row['len_input']) > 250:
        errs['len-'] += 1
        return True, 'len-'
    if (st[0].lower() in row['input'].lower() or
        st[1].lower() in row['input'].lower() or
        st[2].lower() in row['input'].lower() or
        st[3].lower() in row['input'].lower()):
        errs['st in input'] += 1
        return True, 'st in input'
    if pd.isna(row['input']):
        errs['nan'] +=1
        return True, 'nan'
    if len(row['input'].split('\n')) > 1:
        errs['len input'] += 1
        return True, 'len input'
    if row['len_input'] / row['len'] < 0.75:
        errs['len/<0.75'] += 1
        return True, 'len/<0.75'
    if ( row['len_input'] / row['len'] > 2.5):
        errs['len/>2.5'] += 1
        return True, 'len/>2.5'
    pos_score = F.cosine_similarity(prose_emb[i].unsqueeze(0), poem_emb[i].unsqueeze(0))
    if pos_score < 0.6:
        errs['bad_emb'] += 1
        return True, 'bad_emb'
    return False, ''

print(sys_inst)
ind = 0

for i in range(len(df)):
    real_poem = '\n'.join(ast.literal_eval(df.iloc[i]['stanzas']))

    res, reason = check(df.iloc[i], i)
    if res:
        bad += 1
        if bad > ind * 25 and bad <= (ind+1)* 25:
            count += 1
            print()
            print('index', df.iloc[i]['Unnamed: 0'])
            print("REAL:", real_poem)
            #print(reason)

    total += 1

print(f"Count: {count}, all: {bad}")
#print(errs)

The next function collects and structures the model's outputs. It reads the file block by block, extracts the data by looking for `index`, `REAL:`, and `PROSE:` markers, and compiles them into a Python dictionary keyed by the original row index. Any formatting inconsistencies or incomplete generations are automatically flagged during the loop.

In [ ]:
def parse_file(path):
    data = {}

    with open(path, 'r', encoding='utf-8') as f:
        blocks = f.read().strip().split('\n\n')

    for block in blocks:
        lines = block.strip().split('\n')
        if len(lines) < 3:
            continue

        if (not lines[0].startswith("index ") or 
                not lines[1].startswith("REAL:") or 
                not lines[5].startswith("PROSE:")):
            print('!!!', lines)
            continue
        idx = int(lines[0][5:].strip())

        real = []
        prose = None

        mode = None
        for line in lines[1:]:
            if line.startswith("REAL:"):
                mode = "real"
                real.append(line.replace("REAL:", "").strip())
            elif line.startswith("PROSE:"):
                mode = "prose"
                prose = line.replace("PROSE:", "").strip()
            else:
                if mode == "real":
                    real.append(line.strip())
                elif mode == "prose":
                    prose += "\n" + line.strip()

        data[idx] = {
            "real": real,
            "prose": prose
        }

    return data

This step merges the parsed text from `dataset-creation/temp.txt` back into the main DataFrame. To ensure that Gemini's output is mapped to the correct row, the code normalizes and compares the first lines of the original poem from both sources. 

If the text matches, the `input` column is updated with the new prose. Any misalignments, empty results, or indexing errors are captured in the `bad_cases` list for troubleshooting.

In [ ]:
def normalize_line(s):
    return re.sub(r'\s+', ' ', s.strip())

parsed = parse_file("dataset-creation/temp.txt")

bad_cases = []
df2 = pd.read_csv(path, index_col=0)

for idx, item in parsed.items():
    if idx not in df2.index:
        bad_cases.append({
            "idx": idx,
            "reason": "empty",
            "prose": item["prose"],
            "real": real_lines,
            "stanzas": stanzas
        })
        continue

    real_lines = item["real"]
    stanzas = ast.literal_eval(df2.loc[idx, "stanzas"])

    # защита от пустых случаев
    if not real_lines or not stanzas:
        bad_cases.append({
            "idx": idx,
            "reason": "empty",
            "prose": item["prose"],
            "real": real_lines,
            "stanzas": stanzas
        })
        continue

    real_first = normalize_line(real_lines[0])
    stanza_first = normalize_line(stanzas[0])

    if real_first == stanza_first:
        # совпало → заменяем
        df2.loc[idx, "input"] = item["prose"]
    else:
        # не совпало → сохраняем кейс
        bad_cases.append({
            "idx": idx,
            "prose": item["prose"],
            "real_first": real_first,
            "stanza_first": stanza_first,
            "real": real_lines,
            "stanzas": stanzas
        })

The next code prints bad cases.

In [ ]:
print(f"Bad cases: {len(bad_cases)}")

for case in bad_cases[:5]:
    print("\n---")
    print("INDEX:", case["idx"])
    print("PROSE:", case["prose"])
    print("REAL FIRST:", case.get("real_first"))
    print("STANZA FIRST:", case.get("stanza_first"))
    print("REAL:", case["real"])
    print("STANZAS:", case["stanzas"])

Save new version of dataset.

In [ ]:
df2.to_csv(path)